# Linear Algebra
Numpy's "linalg" library is an efficient tool to perform Linear Algebra operations, let's review some in the following sections

In [66]:
import numpy as np #import the library

## 0. Inner product
The inner product between two vectors is given by:
$$\mathbf{x_1}.\mathbf{x_2} = \sum_{i=0}^{n-1}x^{i}_{1} \cdot x^{i}_{2}$$

In [67]:
v = np.array([[4],[2]])
print(f'shape(v) = {np.shape(v)}')
w = np.array([[-1],[2]])
print(f'shape(w) = {np.shape(w)}')

shape(v) = (2, 1)
shape(w) = (2, 1)


In [68]:
#try implementing it from scratch:
def compute_inner(x1:np.ndarray, x2:np.ndarray):
    '''
    :param x1: a vector
    :param x2: a vector
    :return the inner product between x1 and x2
    '''

    return np.sum(x1[:]*x2[:])

inner_scratch = compute_inner(v,w)
print(f'v.w (scratch) = {inner_scratch}')

v.w (scratch) = 0


In [69]:
#or as a lambda function
inner_lambda = lambda x1, x2: np.sum(x1[:]*x2[:])
print(f'v.w (lambda) = {inner_lambda(v,w)}')

v.w (lambda) = 0


In [70]:
#lastly, try the native "np.inner()" method from numpy
inner_np = np.inner(v.ravel(),w.ravel())
print(f'v.w (numpy) = {inner_np}')

v.w (numpy) = 0


Assuming two non-unitary vectors x1 and x2, the angle between them can be computed from the inner product itself: 
$$\cos\theta = \frac{\mathbf{x_{1}}.\mathbf{x_{2}}}{||\mathbf{x_{1}}||.||\mathbf{x_{2}}||},$$

where ||·|| denotes the norm (or size) of the given vector.

In [71]:
def compute_angle(x1:np.ndarray, x2:np.ndarray):
    '''
    :param x1: a vector
    :param x2: a vector
    :return the angle between x1 and x2 (in radians)
    '''

    inner_prod = np.inner(x1.ravel(), x2.ravel()) #compute the inner product between x1 and x2
    x1_norm = np.linalg.norm(x1) #norm of x1
    x2_norm = np.linalg.norm(x2) #norm of x2

    return np.arccos(inner_prod/(x1_norm*x2_norm))

vw_angle = compute_angle(v,w)
print(f'angle between v and w (rad) = {vw_angle}')
#keep in mind that Numpy's trigonometric functions by default return the values in radians. To convert to degrees, simply:
print(f'angle between v and w (degrees) = {np.rad2deg(vw_angle)}')

angle between v and w (rad) = 1.5707963267948966
angle between v and w (degrees) = 90.0


## 1. Transpose of a Matrix
The transpose, denoted by {·}^T, has its columns equal to the rows of the original matrix, and vice-versa. The transpose operator flips the main diagonal such that:
$$\mathbf{A}^{T}_{i,j} = \mathbf{A}_{j,i}$$

Note: an inner product equals to zero means that both vector are perpendicular, or the angle between them is equal to pi (or 90 degrees).

## 2. Matrix Multiplication
When multiplying two matrices A and B, if A has n columns, B must have n rows, as all columns of B are multiplied by A. Assuming A with shape (m,n), and B with shape (n,p), AB will have a shape equal to (m,p). An edge case would be a column vector (n,1) multiplied by a line vector (1,n), which curiously results in a square matrix (n,n)!

In [72]:
def compute_matmul(A:np.ndarray, B:np.ndarray):
    '''
    :param A: a matrix
    :param B: a matrix
    :return the matrix multiplication of A and B
    '''

    #error handling functions are a good way of validating your input data
    #for example, check if the shape between the matrices match to compute the multiplication
    if A.shape[1]!=B.shape[0]:
        raise ValueError(f'[compute_matmul] The number of columns of A is different from the number of rows of B!')
    
    AB = np.zeros(shape=(A.shape[0], B.shape[1])) #shape=(m,p)
    for i in range(0, A.shape[0]):
        AB[i,:] = np.sum(A[i,:]*B.T, axis=1) #linear combination of the row of A with columns of B

    return AB

In [73]:
#case 1: square matrices
A_1 = np.array([[1.,1.],[2.,-1.]])
B_1 = np.array([[2.,2.],[3.,4.]])
AB1 = compute_matmul(A_1, B_1)
print(f'AB1 (scratch) = \n{AB1}')
print()
#now, try it with the built-in method from NumPy
print(f'AB1 (numpy) = \n{A_1@B_1}')

AB1 (scratch) = 
[[5. 6.]
 [1. 0.]]

AB1 (numpy) = 
[[5. 6.]
 [1. 0.]]


In [74]:
#case 2: curious edge case
A_2 = np.array([[0.],[1.],[2.]])
B_2 = np.array([[1.,2.,3.]])
AB2_scratch = compute_matmul(A_2, B_2)
print(f'AB2 (scratch) = \n{AB2_scratch}')
print()
#now, try it with the built-in method from NumPy
print(f'AB2 (numpy) = \n{A_2@B_2}')

AB2 (scratch) = 
[[0. 0. 0.]
 [1. 2. 3.]
 [2. 4. 6.]]

AB2 (numpy) = 
[[0. 0. 0.]
 [1. 2. 3.]
 [2. 4. 6.]]


In [75]:
#case 3: square matrix multiplied by a column vector
A_3 = np.array([[6.,2.,4.],[-1.,4.,3.],[-2.,9.,3.]])
B_3 = np.array([[4.],[-2.],[1.]])
AB3_scratch = compute_matmul(A_3, B_3)
print(f'AB3 (scratch) = \n{AB3_scratch}')
print()
#now, try it with the built-in method from NumPy
print(f'AB3 (numpy) = \n{A_3@B_3}')

AB3 (scratch) = 
[[ 24.]
 [ -9.]
 [-23.]]

AB3 (numpy) = 
[[ 24.]
 [ -9.]
 [-23.]]


## 3. Inverse Matrix
For a square matrix, and only a square matrix, its inverse is such that: 
$$\mathbf{A}^{-1}\mathbf{A}=\mathbf{I},$$
and 
$$\mathbf{A}\mathbf{A}^{-1}=\mathbf{I}.$$ 

However, not all matrices can be inverted! For example, if the determinant is 0, the matrix is called singular, and inverting it is impossible, even if Gauss-Jordan elimination results in "n" pivots. Additionally, a diagonal matrix is invertible only if none of its main diagonal entries are 0.

In [76]:
#one way of computing the inverse is through the Gauss-Jordan elimination method
#here's a VERY slow implementation of the method, just so you get the hang of it

def compute_inverse(A:np.ndarray):
    '''
    :param A: a matrix
    :return the inverse matrix multiplication of A
    '''

    #validate the determinant
    det_A = np.linalg.det(A)
    if det_A == 0:
        raise ValueError(f'[compute_inverse] Singular Matrix!')

    A_copy = np.zeros_like(A) #create a matrix filled with zeros with same shape as A
    A_inv = np.copy(A_copy) #create a matrix that will be updated with the computed inverse

    #build the pivot and inverse placeholder matrices
    for i in range(0, A.shape[0]):
        A_inv[i,i] = 1.
        for j in range(0, A.shape[1]):
            A_copy[i,j] = np.copy(A[i,j])

    #Gauss-Jordan elimination
    for k in range(0,A.shape[0]):
        pivot = np.copy(A_copy[k,k])

        #scale current row by the pivot
        for l in range(0, A.shape[0]):
            A_copy[k,l] /= pivot
            A_inv[k,l] /= pivot

        #eliminate other entries in the column
        for m in range(0, A.shape[1]):
            if m!=k:
                factor = np.copy(A_copy[m,k])
                for n in range(0,A.shape[0]):
                    A_copy[m,n] -= factor*np.copy(A_copy[k,n])
                    A_inv[m,n] -= factor*np.copy(A_inv[k,n])
    return A_inv

In [77]:
#let's try our implementation
F = np.array([[1.,0.,0.],[0.,1.,0.],[0.,-4.,1.]]) #arbitray square matrix
F_inv = compute_inverse(F) #compute the inverse
print(f'F_inv = \n{F_inv}')
print()
print(f'F@F_inv = \n{F@F_inv}') #check if the property is valid

F_inv = 
[[1. 0. 0.]
 [0. 1. 0.]
 [0. 4. 1.]]

F@F_inv = 
[[1. 0. 0.]
 [0. 1. 0.]
 [0. 0. 1.]]


Few... That was a lot! Thankfully, NumPy has a built-in function to compute the inverse :P

In [78]:
np.linalg.inv(F)

array([[ 1.,  0.,  0.],
       [-0.,  1., -0.],
       [ 0.,  4.,  1.]])

In [79]:
F@np.linalg.inv(F)

array([[1., 0., 0.],
       [0., 1., 0.],
       [0., 0., 1.]])

## 3. Transpose of a Matrix
The transpose, denoted by {·}^T, has its columns equal to the rows of the original matrix, and vice-versa. The transpose operator flips the main diagonal such that:
$$\mathbf{A}^{T}_{i,j} = \mathbf{A}_{j,i}$$

In [80]:
def compute_transpose(A:np.ndarray):
    '''
    :param A: a matrix
    :return the transpose of A
    '''

    A_t = np.zeros(shape=(A.shape[1], A.shape[0])) #matrix to store the transpose of A (with flipped dimensions)
    for i in range(0, A.shape[0]):
        for j in range(0, A.shape[1]):
            A_t[j,i] = A[i,j]

    return A_t

In [81]:
A_4 = np.array([[1.,2.,3.],[0.,0.,4.]])
print(f'A_4 = \n{A_4}')
print()
A_4_T = compute_transpose(A_4) 
print(f'A_4_T = \n{A_4_T}')

A_4 = 
[[1. 2. 3.]
 [0. 0. 4.]]

A_4_T = 
[[1. 0.]
 [2. 0.]
 [3. 4.]]


The transpose has a built-it altenative in Numpy as ".T":

In [82]:
A_4.T

array([[1., 0.],
       [2., 0.],
       [3., 4.]])